In [14]:
import os
import shutil
from time import time

import numpy as np
from openfermion import QubitOperator, get_fermion_operator

from ofex.clifford import diagonalizing_clifford, clifford_simulation, tableau_to_pauli, str_tableau, pauli_to_tableau
from ofex.linalg.sparse_tools import expectation, transition_amplitude, apply_operator, state_dot
from ofex.measurement import sorted_insertion, fragment_variance, pauli_split_group, pauli_idx_from_group, \
    pauli_covariance, optimal_sorted_insertion, shot_alloc_by_norm, killer_shift_opt_fermion_hf
from ofex.measurement.iterative_coefficient_splitting import init_ics, run_ics, init_efficient_ics, run_efficient_ics
from ofex.propagator import exact_rte
from ofex.state.chem_ref_state import hf_ground, cisd_ground
from ofex.state.state_tools import to_dense, state_to_fock_vector
from ofex.transforms import fermion_to_qubit_operator, fermion_to_qubit_state
from ofex.utils.chem import molecule_example, run_driver
from test_scripts.random_object import random_state_nparray



## 1. Prepare Hamiltonian and Reference State

Prepare Hamiltonian, reference states and real-time evolution operator of electronic structure of given molecule.

In [15]:
# Obtain the Fermionic Hamiltonian

mol_name = "H4"

mol = molecule_example(mol_name)
mol = run_driver(mol, run_cisd=True)
fham = mol.get_molecular_hamiltonian()
fham = get_fermion_operator(fham)
f_const = fham.constant
fham = fham - f_const

In [16]:
# Fermion-to-Qubit mapping

transform = "symmetry_conserving_bravyi_kitaev"
f2q_kwargs = {"active_fermions": mol.n_electrons,
              "active_orbitals": mol.n_qubits}

n_qubits = mol.n_qubits - 2  # Two-qubit reduction

hf_state_fermion = hf_ground(mol)
hf_state = hf_ground(mol, fermion_to_qubit_map=transform, **f2q_kwargs)
cisd_state = fermion_to_qubit_state(cisd_ground(mol), transform, **f2q_kwargs)

pham = fermion_to_qubit_operator(fham, transform, **f2q_kwargs)
p_const = pham.constant
# Make qubit hamiltonian traceless
pham = pham - p_const

cisd_energy = mol.cisd_energy - p_const - f_const

In [17]:
time_step = 1.0
prop = exact_rte(pham, time_step) # e^{-i H}
time_evolved_state = apply_operator(prop, hf_state)

## 2. Sorted Insertion

- Pauli 1-norm: Expected measurement cost when each Pauli operators are measured individually.
- Sorted Insertion: Pauli grouping technique. If the grouped operators needs to be unitary, set `anticommute=True`. Otherwise, the grouped pauli operators commute to each other and become Clifford-diagonalizable.
- Optimal Sorted Insertion: Allows a Pauli term to be shared across multiple groups. It includes optimization process to reduce the resulting norm.

We can observe the sorted insertion significantly reduce the measurement cost.

In [18]:
pauli_one_norm = pham.induced_norm(order=1)

si_comm = sorted_insertion(pham, anticommute=False)
si_anti = sorted_insertion(pham, anticommute=True)
si_comm_opt = optimal_sorted_insertion(pham, anticommute=False, init_method="even")

si_comm_cost = sum([frag.induced_norm(order=2) for frag in si_comm])
si_anti_cost = sum([frag.induced_norm(order=2) for frag in si_anti])
si_comm_opt_cost = sum([frag.induced_norm(order=2) for frag in si_comm_opt])

print("Pauli One Norm:", pauli_one_norm)
print("Sorted Insertion Cost (commutative):", si_comm_cost)
print("Sorted Insertion Cost (commutative, opt):", si_comm_opt_cost)
print("Sorted Insertion Cost (anticommutative):", si_anti_cost)

Pauli One Norm: 8.54692480320086
Sorted Insertion Cost (commutative): 2.031015101725298
Sorted Insertion Cost (commutative, opt): 1.9937524538014186
Sorted Insertion Cost (anticommutative): 6.00701366295686


For commutative grouping, use `diagonalizing_clifford` for the simultaneous diagonalization of grouped Pauli operators. The example code demonstrate the fragmented measurement of
$$
\braket{\psi|\hat{H}|\psi}=\sum_j \braket{\psi|\hat{H}_j|\psi} = \sum_j \braket{\psi|\hat{C}_j^{\dagger}\hat{Z}_j\hat{C}|\psi},
$$
where $\hat{H}_j$ is a Hamiltonian fragment, $\hat{C}_j$ is corresponding diagonalizing clifford.
Here, $\ket{\psi}$ is randomly generated.

In [19]:
psi = random_state_nparray(n_qubits)
# If psi is sparse, turn on the sparse option for efficient computation.
true_expectation = expectation(pham, psi, sparse=False)
frag_expectation = 0
for j, frag in enumerate(si_comm):
    # Obtain the diagonalized operator and the history of applications of clifford operators.
    tab_z_pauli, coeff, clif_hist = diagonalizing_clifford(frag, n_qubits)
    # Transform the tableau representation to the QubitOperator object.
    z_op = QubitOperator.accumulate(
        tableau_to_pauli(tab_z_pauli, coeff)
    )
    # Obtain C_j|ψ>
    clif_psi = clifford_simulation(psi, clif_hist)

    # Obtain <ψ| C_j† Z_j C_j |ψ>.
    expectation_j = expectation(z_op, clif_psi, sparse=False)

    if j == 1:
        tab_frag, frag_coeff = pauli_to_tableau(frag, n_qubits)
        print("The first fragment:")
        print("Fragment (frag):\n\t" + '\n\t'.join(str(frag).split('\n')))
        print("Tabulated Frag :\n\t" + '\n\t'.join(str_tableau(tab_frag).split('\n')))
        print("Clifford History (clif_hist):", clif_hist)
        print("Tabulated Z Pauli (tab_z_pauli)\n\t" + '\n\t'.join(str_tableau(tab_z_pauli).split('\n')))
        print("Z Operator (z_op):\n\t", '\n\t'.join(str(z_op).split('\n')))

    frag_expectation += expectation_j

# Added print statement for comparison
print("True Expectation:", true_expectation)
print("Fragmented Expectation:", frag_expectation)
print("Difference:", true_expectation - frag_expectation)

The first fragment:
Fragment (frag):
	-0.025454465655117227 [X0 Z1 X2] +
	-0.03872306490340444 [X0 Z1 X3] +
	0.03872306490340444 [X0 Z1 X3 Z4] +
	0.03593469149844004 [X0 Z1 Z4 X5] +
	-0.03593469149844004 [X0 Z1 X5] +
	0.05300619296799148 [X0 X2] +
	0.03872306490340444 [X0 X3] +
	-0.03872306490340444 [X0 X3 Z4] +
	-0.03593469149844004 [X0 Z4 X5] +
	0.03593469149844004 [X0 X5] +
	-0.03593469149844004 [Z1 X2 X3] +
	0.03593469149844004 [Z1 X2 X3 Z4] +
	0.039274121170619215 [Z1 X2 Z4 X5] +
	-0.039274121170619215 [Z1 X2 X5] +
	0.03593469149844004 [X2 X3] +
	-0.03593469149844004 [X2 X3 Z4] +
	-0.039274121170619215 [X2 Z4 X5] +
	0.039274121170619215 [X2 X5] +
	-0.025454465655117227 [X3 Z4 X5] +
	0.05300619296799148 [X3 X5]
Tabulated Frag :
	1 0 0 0 0 0 1 1 1 1 1 1 1 1 0 0 0 0 1 0
	0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
	1 0 1 1 1 1 0 0 0 0 0 0 0 0 1 1 1 1 1 0
	0 1 0 0 0 0 1 1 1 1 0 0 0 0 1 1 1 1 0 1
	0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
	0 1 1 1 1 1 0 0 0 0 1 1 1 1 0 0 0 0 0 1
	0 0 0 0 0 0

## 3. Theoretical Measurement Cost

### Expectation Value Measurement
For each fragment $\hat{H}_j$, $m_j$ shots are allocated. Then, the variance is estimated by the following expression:
$$
V_{tot}=\sum_j \frac{\braket{\psi|\hat{H}_j^2|\psi}-\braket{\psi|\hat{H}_j|\psi}^2}{m_j}.
$$

Note that $m_j \propto \|\hat{H}_j\|_\mathrm{F}=\sqrt{\mathrm{Tr}[\hat{H}_j^2]}$ is a heuristic optimal.

In [20]:
n_frags = len(si_comm)
uniform_shots = np.array([1000] * n_frags)
tot_shots = sum(uniform_shots)

opt_shots = shot_alloc_by_norm(si_comm, tot_shots, transition=False)

v_tot = fragment_variance(si_comm, state1=psi, state2=None, shots=uniform_shots, anticommute=False)
v_tot_opt = fragment_variance(si_comm, state1=psi, state2=None, shots=opt_shots, anticommute=False)

print("Variance with Uniform Shots: ", v_tot)
print("Variance with Optimal Shots: ", v_tot_opt)

Variance with Uniform Shots:  0.0016434572237177165
Variance with Optimal Shots:  0.00041144517340336505


### Transitional Amplitude Measurement

In measurement of transitional amplitude, $\braket{\phi_1|\hat{H}|\phi_2}$, we have two scenarios: fragmented hamiltonian and linear combination of unitaries.
$$
V_{tot} = \sum_j\left(\frac{V_{R,j}}{m_{R,j}}+\frac{V_{I,j}}{m_{I,j}}\right),
$$
where $V_{R,j}$ and $V_{I,j}$ are one-shot variances for real and imaginary part measurements, while $m_{R,j}$ and $m_{I,j}$ are corresponding numbers of shots:
$$
V_{R,j}=0.5(\braket{\phi_1|\hat{H}_j^2|\phi_1}+\braket{\phi_1|\hat{H}_j^2|\phi_1})-\mathrm{Re}[\braket{\phi_1|\hat{H}_j|\phi_2}]^2,
$$
$$
V_{I,j}=0.5(\braket{\phi_1|\hat{H}_j^2|\phi_1}+\braket{\phi_1|\hat{H}_j^2|\phi_1})-\mathrm{Im}[\braket{\phi_1|\hat{H}_j|\phi_2}]^2,
$$
Note that for LCU case, $\hat{H}_j$ is scaled unitary, $\braket{\phi_1|\hat{H}_j^2|\phi_1}=\|\hat{H}_j\|_{\mathrm{F}}^2$ is efficiently calculated instead of the expectation.

Refer to [Arxiv:2409.02504](https://arxiv.org/abs/2409.02504) for the details.

In [21]:
phi_1, phi_2 = random_state_nparray(n_qubits), random_state_nparray(n_qubits)

# If phi_1 and phi_2 are sparse vectors, turning sparse options True makes the computation faster.
# As random_state_nparray is dense, we turned off those options.
true_amp = transition_amplitude(pham, phi_1, phi_2, sparse1=False, sparse2=False)
print("True Amplitude:", true_amp)

# Shot allocation; n_frag × 2 (for real and imaginary parts)
opt_shots_comm = shot_alloc_by_norm(si_comm, tot_shots, transition=True)
opt_shots_anti = shot_alloc_by_norm(si_anti, tot_shots, transition=True)

t = time()
v_amp_comm = fragment_variance(si_comm, state1=phi_1, state2=phi_2, shots=opt_shots_comm, anticommute=False)
t = time() - t
v_amp_anti = fragment_variance(si_anti, state1=phi_1, state2=phi_2, shots=opt_shots_anti, anticommute=True)
print("Transition Amplitude Variance (commutative):", v_amp_comm)
print("Time:", t, "sec")
print("Transition Amplitude Variance (anticommutative):", v_amp_anti)

True Amplitude: (0.10111265977994391+0.11204491629781946j)
Transition Amplitude Variance (commutative): 0.001707208922486969
Time: 0.08840250968933105 sec
Transition Amplitude Variance (anticommutative): 0.014316181995920827


### Buffered Variance Calculation: Pauli Covariance

For variance calculation for large system, we recommend to calculate Pauli Covariance in a parallel manner.
The covariances are buffered in pickle files, which is loaded for efficient calculation in `fragment_variance`.
This is also used in ICS technique which will be explained in this tutorial.

In [22]:
buffer_dir = f'./tmp_measurement_buffer/{mol_name}_{transform}_comm/'
# remove the previous buffer because the states are randomly generated and thus the previous covriances are no longer valid.
if os.path.exists(buffer_dir):
    shutil.rmtree(buffer_dir)
os.makedirs(buffer_dir)
t = time()
pauli_list, grp_pauli_list, pauli_grp_list= pauli_idx_from_group(si_comm)
trans_cov = pauli_covariance(pauli_list, grp_pauli_list,
                             state1=phi_1, state2=phi_2, anticommute=False,
                             num_workers=8, cov_buf_dir=buffer_dir)
print("Time for covariance:", time() - t, "sec")
t = time()
v_amp_comm_cov = fragment_variance(si_comm, state1=phi_1, state2=phi_2, shots=opt_shots_comm, anticommute=False,
                                   true_cov_dict=trans_cov)
print("Time for variance:", time() - t, "sec")  # Computation time is not reduced because the system size is too small.
print("Transition Amplitude Variance (commutative, buffered):", v_amp_comm_cov)


Time for covariance: 1.4828028678894043 sec
Time for variance: 0.09802627563476562 sec
Transition Amplitude Variance (commutative, buffered): 0.0017072089224869733


## 4. Measurement Optimization: Iterative Coefficient Splitting (ICS)

Based on the estimated covariance using classically tractable reference states, such as CISD, ICS optimizes the coefficient split and shot allocation to achieve small measurement cost.

Refer to:

- [Arxiv:2201.01471](https://arxiv.org/abs/2201.01471): ICS for expectation value measurement
- [Arxiv:2409.02504](https://arxiv.org/abs/2409.02504): ICS for transitional amplitude measurement

In [23]:
anticommute = False

# Set ref2=None to run ICS for expectation value measurement.
# If ref2 is given, it runs ICS for the transitional amplitude measurement
si_variance = fragment_variance(si_comm, state1=hf_state, state2=time_evolved_state, shots=opt_shots_comm, anticommute=anticommute)
print("Transition Amplitude Variance (commutative): ", si_variance)

# ICS Optimization
phased_cisd_state = to_dense(cisd_state) * np.exp(-1j * cisd_energy * time_step)
initial_grp, cov_dict = init_ics(pham, ref1=hf_state, ref2=phased_cisd_state,
                                 num_workers=8, cov_buf_dir=None,
                                 anticommute=anticommute)
ics_opt_grp, ics_opt_shot, _, _ = run_ics(pham, initial_grp, cov_dict, transition=True,
                                          conv_atol=1e-5, conv_rtol=1e-3)
ics_opt_shot = ics_opt_shot * tot_shots / np.sum(ics_opt_shot)
ics_variance = fragment_variance(ics_opt_grp, state1=hf_state, state2=time_evolved_state, shots=ics_opt_shot, anticommute=anticommute)
print("Transition Amplitude Variance (commutative, ICS): ", ics_variance)

# Compact ICS
eff_initial_grp = init_efficient_ics(pham, anticommute=anticommute)
eff_ics_opt_grp, _ = run_efficient_ics(pham, eff_initial_grp)
eff_ics_shot = shot_alloc_by_norm(eff_ics_opt_grp, tot_shots, transition=True)
eff_ics_variance = fragment_variance(eff_ics_opt_grp, state1=hf_state, state2=time_evolved_state, shots=eff_ics_shot, anticommute=anticommute)
print("Transition Amplitude Variance (commutative, compact ICS): ", eff_ics_variance)


Transition Amplitude Variance (commutative):  0.003447606671375325
Transition Amplitude Variance (commutative, ICS):  0.0018533776763725053
Transition Amplitude Variance (commutative, compact ICS):  0.003311506381438768


## 5. Killer Shift for QKSD measurement

In calculation of $\braket{HF|\hat{H}|\psi}$, Killer shift finds Hermitian $\hat{T}$ operator such that $\hat{T}\ket{HF}=\tau \ket{HF}$ and minimizes $\|\hat{H}-\hat{T}\|$.
And thus, it turns the full measurement into
$$
\braket{HF|\hat{H}|\psi} = \braket{HF|\hat{H}-\hat{T}|\psi}+\tau\braket{HF|\psi}.
$$
With extended swap test, $\braket{HF|\hat{O}|\psi}$ and $\tau\braket{HF|\psi}$ are simultaneously measureable.
It provides a significant cost reduction in measurement of QKSD matrices.

Refer to:

- [Arxiv:2409.02504](https://arxiv.org/abs/2409.02504): ICS for transitional amplitude measurement

In [25]:
hf_occupation = state_to_fock_vector(hf_state_fermion)
shift_fham, shift_pham, shift_factor = killer_shift_opt_fermion_hf(fham, hf_occupation, transform, optimization_level=1, f2q_kwargs=f2q_kwargs)

shift_si_comm = sorted_insertion(shift_pham, anticommute=False)
shift_si_shot = shot_alloc_by_norm(shift_si_comm, tot_shots, transition=True)

non_shift_tr = transition_amplitude(pham, hf_state, time_evolved_state, sparse1=True, sparse2=False)
shift_tr = transition_amplitude(shift_pham, hf_state, time_evolved_state, sparse1=True, sparse2=False)
overlap = state_dot(hf_state, time_evolved_state)

print("Transition Amplitude (non-shift):", non_shift_tr)
print("Transition Amplitude (shift):", shift_tr + overlap * shift_factor)

non_shift_var = fragment_variance(si_comm, state1=hf_state, state2=time_evolved_state, shots=opt_shots_comm, anticommute=False)
shift_var = fragment_variance(shift_si_comm, state1=hf_state, state2=time_evolved_state, shots=shift_si_shot, anticommute=False)

print("Variance (non-shift):", non_shift_var)
print("Variance (shift):", shift_var)

Transition Amplitude (non-shift): (2.5333764987764753-0.9873672306524451j)
Transition Amplitude (shift): (2.533376498882205-0.9873672306121668j)
Variance (non-shift): 0.003447606671375325
Variance (shift): 0.0001597425637143175
